# trying to analyze the data
main idea:
The analysis should answer three questions in order:

1. Where does Migros currently have strong or weak coverage?
2. Which populated areas have no Migros?
3. Which of those areas have enough population and purchasing power to justify a new store?

In [168]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

In [169]:
# Load and inspect the master data

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

analysis_df = pd.read_csv("data/migros_master_data.csv", dtype={"postal_code": "string"})

print("Shape:", analysis_df.shape)
display(analysis_df.head())

Shape: (3176, 19)


,postal_code,population,avg_income_per_taxpayer,municipality_name,canton_code,aldi_count,coop_count,denner_count,lidl_count,migros_count,external_competitor_count,migros_group_count,postcode_latitude,postcode_longitude,locality_name,complete_location_data,has_migros,population_per_migros,primary_canton
0,1000,4248,57188.176061,Lausanne,VD,0,0,0,0,0,0,0,46.552076,6.686622,Lausanne 25,True,False,4248.000000,VD
1,1003,6879,57188.176061,Lausanne,VD,2,1,1,2,2,5,3,46.520577,6.631877,Lausanne,True,True,3439.500000,VD
2,1004,31463,57188.176061,Lausanne,VD,1,5,1,0,3,6,4,46.528291,6.618649,Lausanne,True,True,10487.666667,VD
3,1005,12454,57188.176061,Lausanne,VD,0,3,0,0,0,3,0,46.520897,6.642754,Lausanne,True,False,12454.000000,VD
4,1006,15621,57188.176061,Lausanne,VD,0,2,2,0,1,2,3,46.512854,6.633803,Lausanne,True,True,15621.000000,VD


In [170]:
# inspect data
print(analysis_df.columns.tolist())
print()
analysis_df.info()

['postal_code', 'population', 'avg_income_per_taxpayer', 'municipality_name', 'canton_code', 'aldi_count', 'coop_count', 'denner_count', 'lidl_count', 'migros_count', 'external_competitor_count', 'migros_group_count', 'postcode_latitude', 'postcode_longitude', 'locality_name', 'complete_location_data', 'has_migros', 'population_per_migros', 'primary_canton']

<class 'pandas.DataFrame'>
RangeIndex: 3176 entries, 0 to 3175
Data columns (total 19 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   postal_code                3176 non-null   string 
 1   population                 3176 non-null   int64  
 2   avg_income_per_taxpayer    3170 non-null   float64
 3   municipality_name          3170 non-null   str    
 4   canton_code                3170 non-null   str    
 5   aldi_count                 3176 non-null   int64  
 6   coop_count                 3176 non-null   int64  
 7   denner_count               3176 n

In [171]:
analysis_df.isna().sum()

postal_code                  0
population                   0
avg_income_per_taxpayer      6
municipality_name            6
canton_code                  6
aldi_count                   0
coop_count                   0
denner_count                 0
lidl_count                   0
migros_count                 0
external_competitor_count    0
migros_group_count           0
postcode_latitude            6
postcode_longitude           6
locality_name                6
complete_location_data       0
has_migros                   0
population_per_migros        0
primary_canton               6
dtype: int64

In [172]:
# Check that every row represents one postcode:
print("Rows:", len(analysis_df))
print(
    "Unique postcodes:",
    analysis_df["postal_code"].nunique()
)

print(
    "Duplicate postcodes:",
    analysis_df["postal_code"].duplicated().sum()
)
# Check total population:
print(
    "Total population:",
    f"{analysis_df['population'].sum():,}"
)

#Check store totals:
store_columns = [
    "migros_count",
    "coop_count",
    "denner_count",
    "aldi_count",
    "lidl_count"
]

analysis_df[store_columns].sum()

Rows: 3176
Unique postcodes: 3176
Duplicate postcodes: 0
Total population: 9,127,125


migros_count    717
coop_count      923
denner_count    724
aldi_count      240
lidl_count      191
dtype: int64

In [173]:
# Exclude only the six postcodes with incomplete income and location data
scoring_df = analysis_df[analysis_df["complete_location_data"] == True].copy()

print("Full dataset:", analysis_df.shape)
print("Scoring dataset:", scoring_df.shape)

Full dataset: (3176, 19)
Scoring dataset: (3170, 19)


In [174]:
analysis_df["complete_location_data"].value_counts(dropna=False)

complete_location_data
True     3170
False       6
Name: count, dtype: int64

In [175]:
scoring_df.isna().sum()

postal_code                  0
population                   0
avg_income_per_taxpayer      0
municipality_name            0
canton_code                  0
aldi_count                   0
coop_count                   0
denner_count                 0
lidl_count                   0
migros_count                 0
external_competitor_count    0
migros_group_count           0
postcode_latitude            0
postcode_longitude           0
locality_name                0
complete_location_data       0
has_migros                   0
population_per_migros        0
primary_canton               0
dtype: int64

# Calculate initial project KPIs

In [176]:
total_population = analysis_df["population"].sum()
total_migros = analysis_df["migros_count"].sum()

postcodes_with_migros = (analysis_df["migros_count"] > 0).sum()

postcodes_without_migros = (analysis_df["migros_count"] == 0).sum()

population_without_migros = analysis_df.loc[analysis_df["migros_count"] == 0,"population"].sum()

population_without_migros_pct = (population_without_migros/ total_population * 100)

print("Total Swiss population:", f"{total_population:,}")
print("Total Migros stores:", total_migros)
print("Postcodes with Migros:", postcodes_with_migros)
print("Postcodes without Migros:", postcodes_without_migros)
print(
    "Population in postcodes without Migros:",
    f"{population_without_migros:,}"
)
print(
    "Population share without Migros:",
    f"{population_without_migros_pct:.2f}%"
)

Total Swiss population: 9,127,125
Total Migros stores: 717
Postcodes with Migros: 549
Postcodes without Migros: 2627
Population in postcodes without Migros: 3,741,584
Population share without Migros: 40.99%


In [177]:
# small summary table - stores by brand
brand_counts_df = pd.DataFrame({
    "brand": [
        "Migros",
        "Coop",
        "Denner",
        "Aldi",
        "Lidl"
    ],
    "number_of_stores": [
        analysis_df["migros_count"].sum(),
        analysis_df["coop_count"].sum(),
        analysis_df["denner_count"].sum(),
        analysis_df["aldi_count"].sum(),
        analysis_df["lidl_count"].sum()
    ]
})

brand_counts_df = brand_counts_df.sort_values(
    "number_of_stores",
    ascending=True
)

brand_counts_df

,brand,number_of_stores
4,Lidl,191
3,Aldi,240
0,Migros,717
2,Denner,724
1,Coop,923


In [178]:
# plotting
brand_colors = {
    "Migros": "#F58220",
    "Coop": "#E30613",
    "Denner": "#19A831",
    "Aldi": "#120DA9",
    "Lidl": "#8B0E70"
}

fig_brand_counts = px.bar(
    brand_counts_df,
    x="number_of_stores",
    y="brand",
    orientation="h",
    color="brand",
    color_discrete_map=brand_colors,
    text="number_of_stores",
    title="Supermarket Locations by Brand in Switzerland"
)

fig_brand_counts.update_traces(
    textposition="outside",
    hovertemplate=(
        "<b>%{y}</b><br>"
        "Stores: %{x:,}"
        "<extra></extra>"
    )
)

fig_brand_counts.update_layout(
    showlegend=False,
    xaxis_title="Number of supermarket locations",
    yaxis_title="Brand",
    template="plotly_white",
    height=450,
    width=900,
    margin=dict(l=80, r=80, t=80, b=60)
)
fig_brand_counts.update_xaxes(tickformat=",")
fig_brand_counts.show()

In [179]:
# Identify postcodes without Migros
no_migros_df = scoring_df[scoring_df["migros_count"] == 0].copy()

# since there was 2 Geneva 
# Create a unique chart label using locality and postcode
no_migros_df["location_label"] = no_migros_df["locality_name"] + " (" + no_migros_df["postal_code"] + ")"

no_migros_df = no_migros_df.sort_values("population",ascending=False)

display(
    no_migros_df[
        [
            "postal_code",
            "locality_name",
            "municipality_name",
            "canton_code",
            "population",
            "avg_income_per_taxpayer",
            "external_competitor_count"
        ]
    ].head(20)
)

,postal_code,locality_name,municipality_name,canton_code,population,avg_income_per_taxpayer,external_competitor_count
129,1202,Genève,"Genève, Pregny-Chambésy",GE,36246,59153.025471,3
570,1920,Martigny,"Martigny, Martigny-Combe, Salvan, Vernayaz",VS,20005,56663.734660,1
150,1226,Thônex,Thônex,GE,16940,62667.582916,1
135,1208,Genève,Genève,GE,16379,58950.440219,0
203,1290,Versoix,"Chavannes-des-Bois, Collex-Bossy, Mies, Versoix","GE, VD",15164,87364.782045,2
3,1005,Lausanne,Lausanne,VD,12454,57188.176061,3
856,3008,Bern,Bern,BE,11463,63636.523330,4
2916,8854,Siebnen,"Galgenen, Schübelbach, Wangen (SZ)",SZ,11369,76149.128060,3
155,1233,Bernex,Bernex,GE,11115,74411.860731,1
38,1052,Le Mont-sur-Lausanne,Le Mont-sur-Lausanne,VD,9790,80137.317266,2


In [180]:
# cleaning the data further
# SELECT THE TOP 15 POPULATION GAPS
top_15_no_migros = no_migros_df.nlargest(15, "population").copy()
top_15_no_migros = top_15_no_migros.sort_values("population", ascending=True).reset_index(drop=True)

display(
    top_15_no_migros[
        [
            "postal_code",
            "locality_name",
            "municipality_name",
            "canton_code",
            "population",
            "avg_income_per_taxpayer",
            "external_competitor_count"
        ]
    ]
)


,postal_code,locality_name,municipality_name,canton_code,population,avg_income_per_taxpayer,external_competitor_count
0,4663,Aarburg,"Aarburg, Boningen, Olten","AG, SO",9049,63019.293937,0
1,5036,Oberentfelden,Oberentfelden,AG,9061,63550.554636,2
2,8623,Wetzikon ZH,"Pfäffikon, Wetzikon (ZH)",ZH,9092,68251.227040,0
3,3097,Liebefeld,"Bern, Köniz",BE,9174,69570.297774,1
4,8802,Kilchberg ZH,Kilchberg (ZH),ZH,9571,141795.142249,1
5,1052,Le Mont-sur-Lausanne,Le Mont-sur-Lausanne,VD,9790,80137.317266,2
6,1233,Bernex,Bernex,GE,11115,74411.860731,1
7,8854,Siebnen,"Galgenen, Schübelbach, Wangen (SZ)",SZ,11369,76149.128060,3
8,3008,Bern,Bern,BE,11463,63636.523330,4
9,1005,Lausanne,Lausanne,VD,12454,57188.176061,3


In [181]:
# Create the first opportunity chart
#top_population_gaps = no_migros_df.nlargest(15,"population").sort_values("population", ascending=True)

fig_population_gaps = px.bar(
    top_15_no_migros,
    x="population",
    y="location_label",
    orientation="h",
    color="avg_income_per_taxpayer",
    color_continuous_scale="Oranges",
    text="population",
    hover_data={
        "postal_code": True,
        "locality_name": False,
        "location_label": False,
        "municipality_name": True,
        "canton_code": True,
        "population": ":,",
        "avg_income_per_taxpayer": ":,.0f",
        "external_competitor_count": True
    },
    labels={
        "population": "Population",
        "location_label": "Locality and postcode",
        "municipality_name": "Municipality",
        "canton_code": "Canton",
        "avg_income_per_taxpayer": "Average income per taxpayer",
        "external_competitor_count": "External competitors"
    },
    title="Largest Population Centres Without a Migros Supermarket"
)

fig_population_gaps.update_traces(
    texttemplate="%{text:,.0f}",
    textposition="outside",
    cliponaxis=False
)

fig_population_gaps.update_layout(
    xaxis_title="Population",
    yaxis_title="Locality and postcode",
    template="plotly_white",
    height=650,
    width=1050,
    coloraxis_colorbar_title="Average income<br>per taxpayer",
    margin=dict(l=160, r=100, t=80, b=60)
)

fig_population_gaps.update_xaxes(tickformat=",")
fig_population_gaps.show()

# Area wise
Switzerland
   ↓
Canton
   ↓
Municipality
   ↓
Locality
   ↓
Postal-code area

# Canton based
Let’s first aggregate the postcode-level data into cantons and build two interactive Plotly charts:

1. Number of Migros stores in each canton.
2. Percentage of Migros versus direct competitors in every canton. 
Migros = Migros supermarkets
Competitors = Coop + Aldi + Lidl

In [182]:
# Some postcodes cover more than one canton. Use the municipality with the largest postcode weight to determine the primary canton.
# Assign every postcode to its main canton
# Confirm that the primary canton column exists in the master data
print("Missing primary cantons:", analysis_df["primary_canton"].isna().sum())
print("Number of cantons:", analysis_df["primary_canton"].nunique())

Missing primary cantons: 6
Number of cantons: 26


In [183]:
## Reuse rows with complete income, municipality, canton and coordinate data
canton_analysis_df = scoring_df.copy()

In [184]:
print("Canton analysis rows:", len(canton_analysis_df))
print("Missing primary cantons:", canton_analysis_df["primary_canton"].isna().sum())
print("Number of cantons:", canton_analysis_df["primary_canton"].nunique())

Canton analysis rows: 3170
Missing primary cantons: 0
Number of cantons: 26


In [185]:
# Add store counts across all postcodes belonging to each canton
canton_store_df = canton_analysis_df.groupby("primary_canton", as_index=False)[["migros_count", "coop_count", "denner_count", "aldi_count", "lidl_count"]].sum()
canton_store_df.head()

,primary_canton,migros_count,coop_count,denner_count,aldi_count,lidl_count
0,AG,42,56,49,15,16
1,AI,1,1,1,0,0
2,AR,4,4,4,1,1
3,BE,72,138,84,23,18
4,BL,25,29,23,6,4


In [186]:
# Direct external competitors are Coop, Aldi and Lidl
canton_store_df["competitor_count"] = canton_store_df["coop_count"] + canton_store_df["aldi_count"] + canton_store_df["lidl_count"]
canton_store_df.head()

,primary_canton,migros_count,coop_count,denner_count,aldi_count,lidl_count,competitor_count
0,AG,42,56,49,15,16,87
1,AI,1,1,1,0,0,1
2,AR,4,4,4,1,1,6
3,BE,72,138,84,23,18,179
4,BL,25,29,23,6,4,39


In [187]:
# total Migros group count
# Denner belongs to Migros Group
canton_store_df["migros_group_count"] = canton_store_df["migros_count"] + canton_store_df["denner_count"]
print(canton_store_df[["migros_count", "coop_count", "denner_count", "aldi_count", "lidl_count"]].sum())

migros_count    717
coop_count      923
denner_count    724
aldi_count      240
lidl_count      191
dtype: int64


In [ ]:
print(canton_store_df[["migros_count", "denner_count", "migros_group_count", "competitor_count"]].sum())